### Extraction, Transform and Load with Duckdb and DLT

In [1]:
import sys
print(sys.executable)

/mnt/c/Users/emman/chicago_employee_data/.venv/bin/python3


In [2]:
import os
import requests
import pandas as pd
import numpy as np
import fastparquet
import pyarrow
from loguru import logger

import dlt
import duckdb
from dlt.common.pipeline import LoadInfo

In [3]:
# Configure loguru to track errors and debug logs

logger.add("chicago_employee.log", rotation="1 MB", level="DEBUG")

1

In [4]:
# Pagination function to get one "page" of data per request and  not full data at once.

def pagination(url):

    while True:
        response = requests.get(url)
        response.raise_for_status()
        yield response.json()

        if "next" not in response.links:
            break
        url = response.links["next"]["url"]

    logger.info(f"Finished fetching all pages from: {url}")


In [5]:
# Chicago Employee API Endpoint

@dlt.resource(table_name="employee")
def get_employee():
    employee_url = "https://data.cityofchicago.org/resource/xzkq-xp2w.json"
    logger.info("Fetch data for Employee Endpoint")
    yield pagination(employee_url)

In [6]:
# Create dlt pipeline and load all data

def assert_load_info(info: LoadInfo, expected_load_packages: int = 1) -> None:
    """Asserts that expected number of packages was loaded and there are no failed jobs"""
    assert len(info.loads_ids) == expected_load_packages
    # all packages loaded
    assert all(package.state == "loaded" for package in info.load_packages) is True
    # no failed jobs in any of the packages
    info.raise_on_failed_jobs()

pipeline = dlt.pipeline(
    pipeline_name='chicago_employee',
    destination='duckdb',
    dataset_name='chicago_employee_data',
    dev_mode=True,
)

load_info = pipeline.run([get_employee()])
logger.info(f"Pipeline finished successfully. Loaded packages: {load_info.loads_ids}")
print(load_info)
assert_load_info(load_info)


/mnt/c/Users/emman/chicago_employee_data/.venv/lib/python3.11/site-packages/dlt/common/configuration/specs/config_providers_context.py:144 DeprecatedImportWarning: The `airflow.operators.python.get_current_context` attribute is deprecated. Please use `'airflow.sdk.get_current_context'`.

2026-03-04 21:39:58.998 | INFO     | __main__:get_employee:6 - Fetch data for Employee Endpoint
2026-03-04 21:40:00.795 | INFO     | __main__:pagination:14 - Finished fetching all pages from: https://data.cityofchicago.org/resource/xzkq-xp2w.json
2026-03-04 21:40:01.879 | INFO     | __main__:<module>:19 - Pipeline finished successfully. Loaded packages: ['1772681998.9703774']


Pipeline chicago_employee load step completed in 0.68 seconds
1 load package(s) were loaded to destination duckdb and into dataset chicago_employee_data_20260305033958
The duckdb destination used duckdb:////mnt/c/Users/emman/chicago_employee_data/1.ETL/chicago_employee.duckdb location to store data
Load package 1772681998.9703774 is LOADED and contains no failed jobs


* Note: Duckdb database (chicago_employee.duckdb) should be created in the working directory. 

In [7]:
# Connect to duckdb database to get first-five rows from employee table

conn = duckdb.connect(f"{pipeline.pipeline_name}.duckdb")

conn.sql(f"SET search_path = '{pipeline.dataset_name}'")

employee_data = conn.sql("SELECT * FROM employee").df()
display(employee_data.head())

,name,job_titles,department,full_or_part_time,salary_or_hourly,annual_salary,_dlt_load_id,_dlt_id,typical_hours,hourly_rate
0,"AARON, JEFFERY M",LIEUTENANT,CHICAGO POLICE DEPARTMENT,F,SALARY,165624,1772681998.9703774,QyimPEwlur4SvQ,NaN,NaN
1,"AARON, KARINA",POLICE OFFICER (ASSIGNED AS DETECTIVE),CHICAGO POLICE DEPARTMENT,F,SALARY,140454,1772681998.9703774,J8wfMPJrIeY6zw,NaN,NaN
2,"ABARCA-COMPTON, RUTH A",ATTORNEY - EXCLUDED,CHICAGO DEPARTMENT OF PUBLIC HEALTH,F,SALARY,131124,1772681998.9703774,V8OFAGhc1ZnfSA,NaN,NaN
3,"ABARCA, EMMANUEL",CONCRETE LABORER,CHICAGO DEPARTMENT OF TRANSPORTATION,F,HOURLY,NaN,1772681998.9703774,lFm8yLNEUeTAbA,40,51.4
4,"ABBASI, ANAS ZAHID Z",SENIOR SECURITY ANALYST,DEPARTMENT OF TECHNOLOGY AND INNOVATION,F,SALARY,120144,1772681998.9703774,SLFlx40RyB6rog,NaN,NaN


In [8]:
# rows and column of employee data

employee_data.shape

(1000, 10)

In [9]:
# employee_data information

employee_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   name               1000 non-null   str  
 1   job_titles         1000 non-null   str  
 2   department         1000 non-null   str  
 3   full_or_part_time  1000 non-null   str  
 4   salary_or_hourly   1000 non-null   str  
 5   annual_salary      808 non-null    str  
 6   _dlt_load_id       1000 non-null   str  
 7   _dlt_id            1000 non-null   str  
 8   typical_hours      192 non-null    str  
 9   hourly_rate        192 non-null    str  
dtypes: str(10)
memory usage: 184.8 KB


In [10]:
# view the created schema 'chicago_employee_data_20260224034242'

conn.sql(f"SET search_path = '{pipeline.dataset_name}'")
display(conn.sql("DESCRIBE"))

┌──────────────────┬──────────────────────────────────────┬─────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────┬───────────┐
│     database     │                schema                │        name         │                                                             column_names                                                              │                                        column_types                                        │ temporary │
│     varchar      │               varchar                │       varchar       │                                                               varchar[]                                                               │                                         varchar[]                                          │  boolean  │
├──────────────────┼───────────

* Perform basic SQL Query

In [11]:
result = conn.sql('SELECT name, job_titles, department, full_or_part_time FROM employee_data ORDER BY annual_salary DESC LIMIT 1').df()
print(result)

                 name                 job_titles  \
0  ANGELES, RONALDO C  AVIATION SECURITY OFFICER   

                       department full_or_part_time  
0  CHICAGO DEPARTMENT OF AVIATION                 F  


### Data Preprocessing

In [12]:
employee_data.columns

Index(['name', 'job_titles', 'department', 'full_or_part_time',
       'salary_or_hourly', 'annual_salary', '_dlt_load_id', '_dlt_id',
       'typical_hours', 'hourly_rate'],
      dtype='str')

In [13]:
# drop these columns in the dataset 

employee_data = employee_data.drop(
    columns=['_dlt_load_id', '_dlt_id']
)

In [14]:
# Get missing data

missing_data = pd.DataFrame(employee_data.isna().sum(), columns=['missing_data'])
missing_data

,missing_data
name,0
job_titles,0
department,0
full_or_part_time,0
salary_or_hourly,0
annual_salary,192
typical_hours,808
hourly_rate,808


In [15]:
num_dtypes = ['annual_salary', 'typical_hours', 'hourly_rate']

cat_dtypes = ['name', 'job_titles', 'department', 'full_or_part_time', 'salary_or_hourly']

***Numerical Columns Wrangling***

In [16]:
# change dtypes from object to numeric and fill missing numbers with mean

for num in num_dtypes:
    if num in employee_data.columns:
        employee_data[num] = pd.to_numeric(employee_data[num])
        employee_data[num] = employee_data[num].fillna(np.mean(employee_data[num]))

print('Numerical Columns Datatypes: \n', employee_data[num_dtypes].dtypes)
print('--'*20, '\n')
print('Numerical Columns Missing Values:\n', employee_data[num_dtypes].isnull().sum())

Numerical Columns Datatypes: 
 annual_salary    float64
typical_hours    float64
hourly_rate      float64
dtype: object
---------------------------------------- 

Numerical Columns Missing Values:
 annual_salary    0
typical_hours    0
hourly_rate      0
dtype: int64


***Categorical Columns Wrangling***

In [17]:
employee_data[cat_dtypes].isnull().sum()

name                 0
job_titles           0
department           0
full_or_part_time    0
salary_or_hourly     0
dtype: int64

In [18]:
employee_data.head()

,name,job_titles,department,full_or_part_time,salary_or_hourly,annual_salary,typical_hours,hourly_rate
0,"AARON, JEFFERY M",LIEUTENANT,CHICAGO POLICE DEPARTMENT,F,SALARY,165624.000000,36.40625,44.250312
1,"AARON, KARINA",POLICE OFFICER (ASSIGNED AS DETECTIVE),CHICAGO POLICE DEPARTMENT,F,SALARY,140454.000000,36.40625,44.250312
2,"ABARCA-COMPTON, RUTH A",ATTORNEY - EXCLUDED,CHICAGO DEPARTMENT OF PUBLIC HEALTH,F,SALARY,131124.000000,36.40625,44.250312
3,"ABARCA, EMMANUEL",CONCRETE LABORER,CHICAGO DEPARTMENT OF TRANSPORTATION,F,HOURLY,111814.149059,40.00000,51.400000
4,"ABBASI, ANAS ZAHID Z",SENIOR SECURITY ANALYST,DEPARTMENT OF TECHNOLOGY AND INNOVATION,F,SALARY,120144.000000,36.40625,44.250312


***Drop Duplicates and Save Clean Data***

In [19]:
employee_data.dtypes

name                     str
job_titles               str
department               str
full_or_part_time        str
salary_or_hourly         str
annual_salary        float64
typical_hours        float64
hourly_rate          float64
dtype: object

In [20]:
employee_data = employee_data.drop_duplicates()

os.makedirs('../clean_data', exist_ok=True)
try:
    employee_data.to_csv('../clean_data/employee_data.csv', index=False)
    print("CSV Data Saved Successfully")
except Exception as e:
    print("Save Failed:", e)

CSV Data Saved Successfully


In [21]:
# Data as a parquet file
try:
    employee_data.to_parquet('../clean_data/employee_data.parquet.brotli', compression='brotli')
    print("Parquet Data Saved Successfully")
except Exception as e:
    print("Save Failed:", e)

Parquet Data Saved Successfully
